# Lesson 09 - Metacognition Design Pattern


## Setup

Dis notebook dey show how to use Metacognition design pattern witin di Microsoft Agent Framework.

**Wetyn you go need:**
- Azure OpenAI deployment wey dem don set wit environment variables
- Azure CLI wey you don log in to (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wetin be Metacognition?

Metacognition na **tinkin about tinkin**. For di context of AI agents, e mean to build agents wey fit:

- **Self-reflect** for dia own outputs and reasoning process
- **Detect errors** and recover well well instead of just failing quietly
- **Evaluate** whether dia answers complete and dey helpful
- **Adapt** dia strategy when first method no work (for example, fallback to backup system)

Metacognitive agent no just dey answer questions — e dey watch how e dey perform and e go adjust quick quick.


## Primary and Backup Tools

One common metacognition pattern na **fallback strategy**. Di agent go try di primary tool first; if e fail (like 404 error), di agent go sabi di failure and sharply switch go di backup tool.

Dis thing dey similar to how real-world systems dey work, where primary services fit no dey available and agents gats self-check di problem before dem select anoda way.

Below na di two flight lookup tools wey we go define:
- **Primary** — e cover Paris, Tokyo, and Barcelona
- **Backup** — e cover Berlin, Sydney, and New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Self-Reflecting Agent wey get Error Recovery

Di agent wey dey below get instruction to try di primary flight system first, sabi when e fail, and quickly kon use di backup system. After e give each answer, e go quickly check itself if e answer di user question well well.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Self-Evaluation Pattern

Narafas wey mean metacognition na **self-evaluation**: another person (or di same person for another time) dey check answer to see if e complete, correct, and useful.

Below we dey create one `ResponseEvaluator` person wey go dey score travel-agent answers for three things.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Summary

For dis lesson, you learn how to build **metacognitive agents** using the Microsoft Agent Framework:

- **Self-reflection**: Agents wey dey monitor their own reasoning and dey communicate wetin happen clearly.
- **Error recovery with fallbacks**: One main tool + backup tool pattern wey the agent go fit detect failure (like 404 errors) and automatically try another source.
- **Self-evaluation**: One separate evaluator agent wey dey score answers for completeness, accuracy, and helpfulness.

These patterns make agents strong, clear, and reliable — important tinz for production deployments.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
